# REINFORCE và PPO tự viết trên CartPole

Bài Intermediate 01 học **giá trị** rồi suy ra policy. Bài này học thẳng **policy** $$\pi_\theta(a \mid s)$$ bằng policy
gradient (mục 5 lý thuyết): REINFORCE không và có baseline, rồi PPO với GAE, nhiều env song song, clipped objective (mục 6).
Cùng CartPole-v1, cùng ngân sách 150 000 bước env, 3 seed mỗi thuật toán, để so sample efficiency và độ ổn định.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm, lớp. Điền vào chỗ `# TODO`. Hàm `gae` dán lại từ bài Basic 03.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) chạy env thật, train, vẽ; test bỏ qua chúng.
- Chấm: `uv run pytest tracks/ai-engineer/modules/07-reinforcement-learning/exercises/intermediate/02-policy-gradient-ppo -q`

In [ ]:
# explore: giới hạn luồng, tìm gốc repo
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
print(ROOT)

In [ ]:
import time

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn

## 1. Actor-critic

`ActorCritic(obs_dim, n_actions, hidden=64)` gồm hai MLP **tách rời** (không chung thân), mỗi cái
`Linear → Tanh → Linear → Tanh → Linear`:

- `actor` ra logits `(B, n_actions)`; `dist(obs)` trả `torch.distributions.Categorical(logits=...)`.
- `critic` ra một số; `value(obs)` trả shape `(B,)` (bỏ chiều cuối, đừng để `(B, 1)`).

Khởi tạo như các cài đặt PPO chuẩn: `nn.init.orthogonal_` cho mọi `weight` với gain $$\sqrt 2$$, bias 0; riêng tầng cuối của
actor gain 0.01 (policy ban đầu gần đều, entropy gần $$\ln n_A$$) và tầng cuối của critic gain 1. Hàm `layer_init` cho sẵn.

In [ ]:
def layer_init(layer: nn.Linear, std: float = 2 ** 0.5) -> nn.Linear:
    nn.init.orthogonal_(layer.weight, std)
    nn.init.zeros_(layer.bias)
    return layer


class ActorCritic(nn.Module):
    def __init__(self, obs_dim: int, n_actions: int, hidden: int = 64):
        super().__init__()
        # TODO: self.actor và self.critic, mỗi cái 3 Linear với Tanh ở giữa, khởi tạo bằng layer_init
        raise NotImplementedError

    def dist(self, obs: torch.Tensor) -> torch.distributions.Categorical:
        # TODO
        raise NotImplementedError

    def value(self, obs: torch.Tensor) -> torch.Tensor:
        # TODO: shape (B,)
        raise NotImplementedError

## 2. Loss

- `reinforce_loss(logp, returns, baseline=None)`: $$-\frac1N\sum_t \log\pi(a_t \mid s_t)\,(G_t - b_t)$$. Trọng số
  $$G_t - b_t$$ là hằng số với gradient của policy: không để gradient chảy qua `baseline` (critic học bằng loss riêng).
- `ppo_policy_loss(new_logp, old_logp, adv, clip_eps)`: với $$\rho = \exp(\text{new} - \text{old})$$, loss
  $$-\frac1N\sum \min\big(\rho A,\ \mathrm{clip}(\rho, 1-\epsilon, 1+\epsilon)A\big)$$. Trả `(loss, stats)` với
  `stats = {"clipfrac": tỉ lệ mẫu có |ρ − 1| > ε, "approx_kl": trung bình của (ρ − 1) − log ρ}`, cả hai là `float`.
- `gae(...)`: **dán nguyên** hàm của bài Basic 03 (numpy, cùng quy ước buffer).

In [ ]:
def reinforce_loss(logp: torch.Tensor, returns: torch.Tensor, baseline: torch.Tensor | None = None) -> torch.Tensor:
    # TODO
    raise NotImplementedError


def ppo_policy_loss(new_logp: torch.Tensor, old_logp: torch.Tensor, adv: torch.Tensor,
                    clip_eps: float) -> tuple[torch.Tensor, dict]:
    # TODO: trả (loss, {"clipfrac": float, "approx_kl": float})
    raise NotImplementedError


def gae(rewards, values, next_values, terminated, truncated, gamma, lam):
    # TODO: dán hàm gae của bài Basic 03
    raise NotImplementedError

## 3. REINFORCE

`train_reinforce(make_env, total_steps, seed, baseline=True, gamma=0.99, lr=1e-3, episodes_per_batch=8)`:

1. `torch.manual_seed(seed)`; một env; `ActorCritic`; Adam với `lr`. Episode thứ k (đếm từ 0) reset với `seed + k`.
2. Lặp tới khi tổng số bước env đã chạy ≥ `total_steps`: chạy trọn `episodes_per_batch` episode bằng cách **lấy mẫu** action
   từ `model.dist`, ghi log-prob của action đã lấy; tính reward-to-go $$G_t$$ (có chiết khấu) cho từng episode.
3. Gom cả batch: loss = `reinforce_loss(logp, G, V)` + `0.5 * MSE(V, G)` khi `baseline=True` (V là `model.value` của các
   observation), chỉ `reinforce_loss(logp, G)` khi không có baseline. Một bước Adam mỗi batch.

Trả `{"returns": [return từng episode], "end_steps": [tổng số bước env khi episode đó kết thúc], "model": model}`.

In [ ]:
def train_reinforce(make_env, total_steps: int, seed: int, baseline: bool = True, gamma: float = 0.99,
                    lr: float = 1e-3, episodes_per_batch: int = 8) -> dict:
    # TODO: theo 3 bước ở trên
    raise NotImplementedError

## 4. PPO: thu rollout

`make_vec_env(make_env, n_envs)` (cho sẵn) tạo `SyncVectorEnv` chế độ **SAME_STEP autoreset**: khi một env kết thúc ở bước
$$t$$, observation trả về đã là observation sau reset, còn observation thật cuối episode nằm ở `info["final_obs"][i]`.

`collect_rollout(envs, model, obs, n_steps)` chạy `n_steps` bước trên `N` env, lấy mẫu action từ `model.dist` (không ghi graph
autograd), trả `(buf, obs)` với `obs` là observation để bắt đầu rollout sau và `buf` là dict numpy:

| Khoá | Shape | Nội dung |
| --- | --- | --- |
| `obs` | `(T, N, obs_dim)` float32 | observation trước action |
| `actions` | `(T, N)` int64 | |
| `logp` | `(T, N)` float32 | log-prob của action lúc thu (là "old" của PPO) |
| `rewards`, `values`, `next_values` | `(T, N)` float64 | `next_values` = $$V$$ của observation **thật** sau bước (dùng `final_obs` khi env vừa kết thúc) |
| `terminated`, `truncated` | `(T, N)` bool | |
| `episode_returns` | list float | tổng reward của các episode kết thúc trong rollout này (theo thứ tự bước, rồi env) |

Tổng reward đang dở của từng env phải sống qua các lần gọi: lưu ở thuộc tính `envs.ep_returns` (mảng `(N,)`, tạo nếu chưa có).

In [ ]:
def make_vec_env(make_env, n_envs: int) -> gym.vector.SyncVectorEnv:
    return gym.vector.SyncVectorEnv([make_env] * n_envs, autoreset_mode=gym.vector.AutoresetMode.SAME_STEP)


def collect_rollout(envs, model: ActorCritic, obs: np.ndarray, n_steps: int) -> tuple[dict, np.ndarray]:
    # TODO: trả (buf, obs_tiếp_theo) theo bảng trên
    raise NotImplementedError

## 5. PPO: cập nhật và vòng train

`ppo_update(model, opt, batch, rng, epochs=4, n_minibatches=4, clip_eps=0.2, vf_coef=0.5, ent_coef=0.01, max_grad_norm=0.5)`:
`batch` là dict tensor đã làm phẳng thành `B = T·N` mẫu với khoá `obs`, `actions`, `logp`, `advantages`, `returns`. Mỗi epoch
xáo bằng `rng.permutation(B)`, chia `n_minibatches` minibatch bằng nhau; mỗi minibatch: chuẩn hoá advantage **trong
minibatch** (trừ mean, chia `adv.std()` của torch, mặc định không lệch, cộng 1e-8), loss = `ppo_policy_loss` + `vf_coef` · $$\tfrac12$$MSE(V, returns) − `ent_coef` ·
entropy trung bình, clip norm gradient, một bước optimizer. Trả `stats` của minibatch cuối cộng `"entropy"` và `"value_loss"`.

`train_ppo(make_env, total_steps, seed, n_envs=4, n_steps=128, lr=2.5e-4, gamma=0.99, lam=0.95, **update_kwargs)`:
`torch.manual_seed(seed)`, `rng = np.random.default_rng(seed)`, `envs.reset(seed=seed)`, Adam (`eps=1e-5`), lặp
`total_steps // (n_envs · n_steps)` lần: lr giảm tuyến tính từ `lr` về 0 (đặt ở đầu mỗi lần lặp: `lr · (1 − k / số_lần)`),
`collect_rollout`, `gae`, `ppo_update`. Trả `{"returns", "end_steps", "stats", "model"}`: `end_steps[i]` là tổng số bước env
(tính cả N env) ở cuối rollout có episode `i` kết thúc; `stats` một dict mỗi lần lặp, có thêm khoá `"step"`.

In [ ]:
def ppo_update(model: ActorCritic, opt: torch.optim.Optimizer, batch: dict, rng: np.random.Generator, epochs: int = 4,
               n_minibatches: int = 4, clip_eps: float = 0.2, vf_coef: float = 0.5, ent_coef: float = 0.01,
               max_grad_norm: float = 0.5) -> dict:
    # TODO
    raise NotImplementedError


def train_ppo(make_env, total_steps: int, seed: int, n_envs: int = 4, n_steps: int = 128, lr: float = 2.5e-4,
              gamma: float = 0.99, lam: float = 0.95, **update_kwargs) -> dict:
    # TODO
    raise NotImplementedError

In [ ]:
# explore: kiểm nhanh trước khi train lâu: policy ban đầu gần đều, entropy gần ln 2, value gần 0
make_env = lambda: gym.make("CartPole-v1")
torch.manual_seed(0)
probe = ActorCritic(4, 2)
o = torch.as_tensor(np.stack([make_env().reset(seed=i)[0] for i in range(256)]))
print(f"entropy ban đầu {probe.dist(o).entropy().mean():.4f} | ln 2 = {np.log(2):.4f} | |V| trung bình {probe.value(o).abs().mean():.3f}")

In [ ]:
# explore: train 3 thuật toán x 3 seed, 150 000 bước env mỗi lần; cả cell khoảng 6-10 phút trên CPU
torch.set_num_threads(4)
ALGOS = {
    "REINFORCE": lambda seed: train_reinforce(make_env, 150_000, seed, baseline=False),
    "REINFORCE + baseline": lambda seed: train_reinforce(make_env, 150_000, seed, baseline=True),
    "PPO": lambda seed: train_ppo(make_env, 150_000, seed),
}
runs = {}
for name, fn in ALGOS.items():
    for seed in [0, 1, 2]:
        start = time.perf_counter()
        runs[name, seed] = fn(seed)
        r = np.array(runs[name, seed]["returns"])
        print(f"{name:21s} seed {seed}: {time.perf_counter() - start:4.0f} s | {len(r)} episode | TB 20 episode cuối {r[-20:].mean():6.1f}")

In [ ]:
# explore: đường cong học theo số bước env, và bảng return trung bình theo cửa sổ 10 000 bước
import matplotlib.pyplot as plt

def window_mean(h, upto, width=10_000):
    r, s = np.array(h["returns"]), np.array(h["end_steps"])
    sel = r[(s > upto - width) & (s <= upto)]
    return sel.mean() if len(sel) else float("nan")

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5), sharey=True)
for ax, name in zip(axes, ALGOS):
    for seed in [0, 1, 2]:
        h = runs[name, seed]
        grid = np.arange(10_000, 150_001, 5_000)
        ax.plot(grid, [window_mean(h, g) for g in grid], label=f"seed {seed}")
    ax.set_title(name); ax.set_xlabel("bước env")
axes[0].set_ylabel("return (TB cửa sổ 10k bước)"); axes[0].legend()
plt.show()

print(f"{'thuật toán':21s} | {'50k':>17s} | {'100k':>17s} | {'150k':>17s}")
for name in ALGOS:
    cols = [" / ".join(f"{window_mean(runs[name, s], k):5.0f}" for s in [0, 1, 2]) for k in [50_000, 100_000, 150_000]]
    print(f"{name:21s} | " + " | ".join(f"{c:>17s}" for c in cols))

In [ ]:
# explore: chẩn đoán PPO theo từng lần cập nhật (seed 0): approx KL, clip fraction, entropy
st = runs["PPO", 0]["stats"]
steps = [s["step"] for s in st]
fig, axes = plt.subplots(1, 3, figsize=(15, 3))
for ax, key in zip(axes, ["approx_kl", "clipfrac", "entropy"]):
    ax.plot(steps, [s[key] for s in st]); ax.set_title(key); ax.set_xlabel("bước env")
plt.show()
print("approx_kl lớn nhất:", round(max(s["approx_kl"] for s in st), 4), "| clipfrac lớn nhất:", round(max(s["clipfrac"] for s in st), 3),
      "| entropy cuối:", round(st[-1]["entropy"], 3))

In [ ]:
# explore: đánh giá cuối: policy greedy (argmax) và policy lấy mẫu, 20 episode cố định (seed 10 000..10 019)
def run_policy(model, greedy, n=20, seed=10_000):
    env, out = make_env(), []
    for i in range(n):
        obs, _ = env.reset(seed=seed + i)
        done, total = False, 0.0
        while not done:
            with torch.no_grad():
                d = model.dist(torch.as_tensor(obs).unsqueeze(0))
            a = int(d.probs.argmax()) if greedy else int(d.sample())
            obs, r, te, tr, _ = env.step(a)
            total += r
            done = te or tr
        out.append(total)
    return np.mean(out)

torch.manual_seed(0)
for name in ALGOS:
    g = [run_policy(runs[name, s]["model"], True) for s in [0, 1, 2]]
    smp = [run_policy(runs[name, s]["model"], False) for s in [0, 1, 2]]
    print(f"{name:21s} | greedy {' / '.join(f'{v:5.1f}' for v in g)} | lấy mẫu {' / '.join(f'{v:5.1f}' for v in smp)}")

## 6. Đọc kết quả

So với bảng mục 5, 6 lý thuyết. Câu hỏi cần trả lời bằng số của bạn: baseline giảm phương sai tới mức nào (độ chênh giữa 3 seed,
độ "răng cưa" của đường cong); PPO dùng mỗi mẫu nhiều lần (4 epoch) và có GAE thì tới 500 sớm hơn REINFORCE bao nhiêu bước;
approx KL và clip fraction có nằm trong vùng an toàn (KL dưới khoảng 0.02, clip fraction dưới khoảng 0.2) không.

Sau khi chạy hết notebook, làm phần **trắc nghiệm** của bài trong app.